# 📊 Equity Performance & Risk Analysis (WRDS)

## 👥 Objective
This project analyzes stock performance, risk, and diversification using WRDS (CRSP data).

## ⚠️ Important Note
The selected stocks (AAPL, GOOGL, MSFT) are examples only.  
Users can freely change the tickers to analyze any stocks.

## 📦 Data Source
- WRDS CRSP Daily Stock File

## 🧩 Workflow
1. Import Libraries
2. Connect to WRDS
3. User Input
4. Data Extraction
5. Data Processing
6. Performance Analysis
7. Risk Analysis
8. Advanced Metrics
9. Conclusion

## 🧩 Step 1: Import Libraries

We import libraries for data processing, visualization, and database access.

- pandas & numpy → data handling  
- matplotlib & seaborn → visualization  
- wrds → database connection  

These tools support the entire analysis workflow.

In [ ]:
import wrds
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 🧩 Step 2: Connect to WRDS

We connect to WRDS to access CRSP financial

In [ ]:
username = "xinyisong"
db = wrds.Connection(wrds_username=username)

## 🧩 Step 3: User Input

Users can customize:

- Stock tickers  
- Time period  

📌 Example stocks are used for demonstration only.

In [ ]:
# User can modify this freely
tickers = ["AAPL", "GOOGL", "MSFT"]

# Example alternative:
# tickers = ["TSLA", "NVDA", "AMZN"]

start_year = 2018
end_year = 2023

## 🧩 Step 4: Data Extraction

We retrieve daily stock prices and returns from CRSP.

In [ ]:
def get_stock_data(db, ticker):
    query = f"""
        SELECT a.date, a.prc, a.ret
        FROM crsp.dsf a
        JOIN crsp.stocknames b
        ON a.permno = b.permno
        WHERE b.ticker = '{ticker}'
    """
    
    df = db.raw_sql(query)
    df['date'] = pd.to_datetime(df['date'])
    return df

stock_data = {}

for t in tickers:
    df = get_stock_data(db, t)
    
    df = df[(df['date'] >= f"{start_year}-01-01") &
            (df['date'] <= f"{end_year}-12-31")]
    
    stock_data[t] = df

stock_data[tickers[0]].head()

## 🧩 Step 5: Normalization

We rebase prices to compare relative performance.

In [ ]:
for t in stock_data:
    df = stock_data[t]
    df['normalized'] = df['prc'] / df['prc'].iloc[0]

## 🧩 Step 6: Market Benchmark

We use S&P 500 as the benchmark.

In [ ]:
query = """
    SELECT date, vwretd
    FROM crsp.dsi
"""

index_df = db.raw_sql(query)
index_df['date'] = pd.to_datetime(index_df['date'])

index_df = index_df[
    (index_df['date'] >= f"{start_year}-01-01") &
    (index_df['date'] <= f"{end_year}-12-31")
]

index_df['cum_return'] = (1 + index_df['vwretd']).cumprod()

## 📊 Rebased Performance

This chart compares normalized stock prices with the market benchmark (S&P 500).

📌 Interpretation:
- If a stock line is above the market → it outperforms  
- If below → it underperforms  

💡 Why it matters:
This helps investors quickly identify which stocks generate superior returns relative to the market.

In [ ]:
plt.figure(figsize=(12,6))

for t in stock_data:
    plt.plot(stock_data[t]['date'],
             stock_data[t]['normalized'],
             label=t)

plt.plot(index_df['date'],
         index_df['cum_return'],
         label='S&P500',
         linestyle='--')

plt.title("Rebased Performance")
plt.legend()
plt.show()

## 📈 Cumulative Return

This chart shows how an investment grows over time.

📌 Interpretation:
- A steeper curve indicates stronger performance  
- Allows comparison of long-term profitability  

💡 Why it matters:
Investors care about total wealth growth, not just daily returns.

In [ ]:
plt.figure(figsize=(12,6))

for t in stock_data:
    cum = (1 + stock_data[t]['ret']).cumprod()
    plt.plot(stock_data[t]['date'], cum, label=t)

plt.title("Cumulative Return")
plt.legend()
plt.show()

## 📉 Volatility (Risk)

Volatility measures the variability of returns.

📌 Interpretation:
- Higher volatility → higher risk  
- Lower volatility → more stable returns  

💡 Why it matters:
Understanding risk is essential for making informed investment decisions.

In [ ]:
plt.figure(figsize=(12,6))

for t in stock_data:
    vol = stock_data[t]['ret'].rolling(30).std()
    plt.plot(stock_data[t]['date'], vol, label=t)

plt.title("Volatility (30-day)")
plt.legend()
plt.show()

## 📉 Drawdown

Drawdown measures the decline from a historical peak.

📌 Interpretation:
- Larger drawdown → greater potential loss  
- Indicates downside risk during market downturns  

💡 Why it matters:
Investors are often more concerned about losses than gains.

In [ ]:
plt.figure(figsize=(12,6))

for t in stock_data:
    df = stock_data[t]
    
    cum = (1 + df['ret']).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    
    plt.plot(df['date'], dd, label=t)

plt.title("Drawdown")
plt.legend()
plt.show()

## 🔗 Correlation

This heatmap shows the relationship between stock returns.

📌 Interpretation:
- High correlation → stocks move together  
- Low correlation → better diversification  

💡 Why it matters:
Diversification reduces portfolio risk.

In [ ]:
returns = pd.DataFrame()

for t in stock_data:
    returns[t] = stock_data[t].set_index('date')['ret']

sns.heatmap(returns.corr(), annot=True)
plt.title("Correlation")
plt.show()

## ⚖️ Sharpe Ratio

The Sharpe ratio measures return per unit of risk.

📌 Interpretation:
- Higher Sharpe → more efficient investment  
- Lower Sharpe → less reward for risk  

💡 Why it matters:
Not all high-return investments are attractive once risk is considered.

In [ ]:
sharpe = {}

for t in stock_data:
    r = stock_data[t]['ret']
    sharpe[t] = (r.mean() / r.std()) * np.sqrt(252)

pd.Series(sharpe)

## 📊 Alpha & Beta (CAPM)

We evaluate stock performance relative to the market.

📌 Interpretation:
- Beta > 1 → more sensitive to market movements  
- Beta < 1 → less sensitive  
- Alpha > 0 → outperforms the market  

💡 Why it matters:
This helps distinguish between market-driven returns and stock-specific performance.

In [ ]:
import statsmodels.api as sm

alpha_beta = {}

for t in stock_data:
    df = stock_data[t].copy()
    
    merged = pd.merge(df, index_df, on="date", how="inner")

    merged = merged[['ret', 'vwretd']]

    merged['ret'] = pd.to_numeric(merged['ret'], errors='coerce')
    merged['vwretd'] = pd.to_numeric(merged['vwretd'], errors='coerce')

    merged = merged.replace([np.inf, -np.inf], np.nan)

    merged = merged.dropna()

    merged = merged.astype(float)

    if len(merged) < 30:
        print(f"{t} skipped (not enough data)")
        continue

    X = sm.add_constant(merged['vwretd'])
    y = merged['ret']

    model = sm.OLS(y, X).fit()

    alpha_beta[t] = {
        "alpha": model.params['const'],
        "beta": model.params['vwretd']
    }

pd.DataFrame(alpha_beta).T

## 📊 Rolling Sharpe Ratio

This shows how risk-adjusted performance evolves over time.

📌 Interpretation:
- Stable values → consistent performance  
- Fluctuating values → unstable performance  

💡 Why it matters:
Investors prefer stable and predictable returns.

In [ ]:
plt.figure(figsize=(12,6))

for t in stock_data:
    r = stock_data[t]['ret']
    
    rs = (r.rolling(60).mean() /
          r.rolling(60).std()) * np.sqrt(252)
    
    plt.plot(stock_data[t]['date'], rs, label=t)

plt.title("Rolling Sharpe Ratio")
plt.legend()
plt.show()

## 📉 Maximum Drawdown Summary

This summarizes the worst loss for each stock.

📌 Interpretation:
- Larger negative values → higher downside risk  

💡 Why it matters:
Helps investors understand worst-case scenarios.

In [ ]:
max_dd = {}

for t in stock_data:
    df = stock_data[t]
    
    cum = (1 + df['ret']).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    
    max_dd[t] = dd.min()

pd.Series(max_dd)

## 🔍 Key Findings

- Some stocks outperform the market significantly  
- Higher returns often come with higher volatility  
- Drawdowns reveal hidden downside risks  
- Correlation suggests diversification opportunities  
- Sharpe ratio shows true risk-adjusted performance

## ⚠️ Limitations

- Based on historical data  
- No transaction costs included  
- Limited number of stocks  
- Market assumptions may not hold

## ✅ Conclusion

This project evaluates stocks using:

- Performance metrics  
- Risk measures  
- Market benchmarks  

📌 Key takeaway:
Investors should consider both return and risk when making decisions.

The framework is flexible and can be applied to any stocks